# Reguły Asocjacyjne

<div align="center">

### Podpunkt 1 

</div>


In [3]:
import numpy as np
import pandas as pd

df = pd.read_csv('market.csv', sep=';')

data = {e: 0 for e in list(df.columns)}

s = 0
for (index, row_data) in df.iterrows():
    s += row_data.where(row_data == 1).count()

print(s / df.shape[0])



8.786637931034482


Komentarz:

Przykładowo, jak kupują masło mięso, to tez często kupują ser. (Ma to sens, bo z tego mozna przykładowo złozyć burgera, albo inny obiad). Pytanie jest jeszcze dotyczące jak wybierać support (nośnik, wsparcie). Nie ma jednego dobrego sposobu, najpierw trzeba przyjrzeć się danym. Przykładowo tutaj, jeden koszyk (tranzakcja) ma średnio 9 produktów kupionych spośród 21, czyli jest gęsto zapełniony. Mozna sprawdzić nośnik, dla kazdego produktu osobno, w sumie będzie to jak czesto występuje przy zakupie. I wiemy, ze próg na pewno nie moze być powyzej, bo nie znajdziemy zadnej reguły.

In [ ]:
from __future__ import annotations

import csv
from collections import defaultdict
from itertools import combinations
from typing import Dict, FrozenSet, List, Set, Tuple
import time
import random
from dataclasses import dataclass

Item = str
Itemset = FrozenSet[Item]


def load_market_csv(path: str, sep: str = ";") -> List[Itemset]:
    """
    Wczytuje format jak w market.csv:
    - 1 kolumna
    - header: "Bread;Honey;...;Sugar"
    - wiersze: "0;1;0;...;1" (obecność produktu)
    Zwraca listę transakcji jako frozenset nazw produktów.
    """
    transactions: List[Itemset] = []

    with open(path, "r", newline="", encoding="utf-8") as f:
        reader = csv.reader(f)
        header_row = next(reader)
        if not header_row:
            raise ValueError("Brak nagłówka w CSV.")

        # Cały nagłówek siedzi w jednej komórce
        items = header_row[0].split(sep)
        n_items = len(items)

        for row in reader:
            if not row:
                continue
            bits = row[0].split(sep)
            if len(bits) != n_items:
                raise ValueError(f"Niezgodna liczba kolumn: {len(bits)} != {n_items}")

            basket = frozenset(items[i] for i, v in enumerate(bits) if v.strip() == "1")
            transactions.append(basket)

    return transactions


def _count_singletons(transactions: List[Itemset]) -> Dict[Itemset, int]:
    counts: Dict[Itemset, int] = defaultdict(int)
    for t in transactions:
        for item in t:
            counts[frozenset([item])] += 1
    return dict(counts)


def _apriori_gen(L_prev: Set[Itemset], k: int) -> Set[Itemset]:
    """
    Generowanie kandydatów Ck z L_{k-1}:
    - join step: łączymy zbiory (k-1)-elementowe, które mają wspólny prefix długości k-2
    - prune step: odrzucamy kandydata, jeśli któryś (k-1)-podzbiór nie jest w L_prev
    """
    # Zamieniamy na posortowane krotki, żeby móc robić "prefix join"
    L_list: List[Tuple[Item, ...]] = sorted(tuple(sorted(s)) for s in L_prev)

    Ck: Set[Itemset] = set()
    m = len(L_list)

    for i in range(m):
        for j in range(i + 1, m):
            a = L_list[i]
            b = L_list[j]

            # Prefix długości k-2 musi być taki sam
            if a[: k - 2] != b[: k - 2]:
                break  # dalej b będą miały już inny prefix (bo L_list jest sortowane)

            # Kandydat to unia (powinien mieć rozmiar k)
            cand = frozenset(a) | frozenset(b)
            if len(cand) != k:
                continue

            # Prune: wszystkie (k-1)-podzbiory kandydata muszą być częste
            ok = True
            cand_sorted = tuple(sorted(cand))
            for subset in combinations(cand_sorted, k - 1):
                if frozenset(subset) not in L_prev:
                    ok = False
                    break

            if ok:
                Ck.add(cand)

    return Ck


def apriori(
    transactions: List[Itemset],
    min_support: float | None = None,
    min_support_count: int | None = None,
) -> Dict[Itemset, int]:
    """
    Zwraca słownik: częsty_zbiór_przedmiotów -> liczba transakcji (support count)

    Ustawiasz próg wsparcia:
    - albo min_support w [0,1], np. 0.30
    - albo min_support_count jako liczba transakcji, np. 139
    """
    n = len(transactions)
    if n == 0:
        return {}

    if (min_support is None) == (min_support_count is None):
        raise ValueError("Podaj dokładnie jedno z: min_support albo min_support_count.")

    if min_support_count is None:
        if not (0.0 < min_support <= 1.0):
            raise ValueError("min_support musi być w (0,1].")
        min_support_count = int(min_support * n + 1e-9)  # bez zaokrąglania w górę

    # L1
    counts_1 = _count_singletons(transactions)
    Lk: Set[Itemset] = {s for s, c in counts_1.items() if c >= min_support_count}
    frequent_counts: Dict[Itemset, int] = {s: counts_1[s] for s in Lk}

    k = 2
    while Lk:
        Ck = _apriori_gen(Lk, k)
        if not Ck:
            break

        # Zliczanie wsparcia kandydatów:
        # optymalizacja: generujemy k-elementowe kombinacje z transakcji i sprawdzamy czy są w Ck.
        Ck_counts: Dict[Itemset, int] = defaultdict(int)

        # Zbiór elementów, które w ogóle występują w Lk (ogranicza kombinacje)
        active_items: Set[Item] = set()
        for s in Lk:
            active_items.update(s)

        Ck_lookup = Ck 

        for t in transactions:
            t_filtered = sorted([x for x in t if x in active_items])
            if len(t_filtered) < k:
                continue

            for comb in combinations(t_filtered, k):
                cand = frozenset(comb)
                if cand in Ck_lookup:
                    Ck_counts[cand] += 1

        # Lk = {c in Ck: count >= min_support_count}
        Lk = {c for c, cnt in Ck_counts.items() if cnt >= min_support_count}
        for s in Lk:
            frequent_counts[s] = Ck_counts[s]

        k += 1

    return frequent_counts

@dataclass(frozen=True)
class Rule:
    antecedent: Itemset
    consequent: Itemset
    support: float        # support(X ∪ Y)
    confidence: float     # support(X ∪ Y) / support(X)
    lift: float           # confidence / support(Y)
    leverage: float       # support(X ∪ Y) - support(X)*support(Y)


def generate_association_rules(
    freq_counts: Dict[Itemset, int],
    n_transactions: int,
    min_support: float = 0.0,
    min_confidence: float = 0.0,
) -> List[Rule]:
    """
    Generuje reguły X -> Y z częstych zbiorów (Apriori output).
    - freq_counts: itemset -> support count
    - n_transactions: liczba transakcji
    - min_support: próg support dla reguły (support(X∪Y))
    - min_confidence: próg confidence

    Wymagania:
    - freq_counts musi zawierać wsparcia dla wszystkich podzbiorów,
      które pojawiają się jako antecedent (X) i consequent (Y).
      W praktyce: jeśli Apriori zwraca wszystkie częste zbiory, to zwykle jest OK.
    """
    if n_transactions <= 0:
        raise ValueError("n_transactions musi być > 0")

    rules: List[Rule] = []

    # iterujemy po częstych zbiorach S o rozmiarze >= 2
    for itemset, sup_count_xy in freq_counts.items():
        if len(itemset) < 2:
            continue

        support_xy = sup_count_xy / n_transactions
        if support_xy < min_support:
            continue

        items_sorted = tuple(sorted(itemset))

        # wszystkie niepuste, właściwe podzbiory jako antecedent X
        for r in range(1, len(items_sorted)):
            for X_tuple in combinations(items_sorted, r):
                X = frozenset(X_tuple)
                Y = itemset - X
                if not Y:
                    continue

                # potrzebujemy support(X)
                sup_count_x = freq_counts.get(X)
                if sup_count_x is None:
                    # jeśli X nie jest w freq_counts (np. bo nie było częste), nie policzysz confidence poprawnie
                    continue

                support_x = sup_count_x / n_transactions
                confidence = support_xy / support_x

                if confidence < min_confidence:
                    continue

                # lift i leverage wymagają support(Y)
                sup_count_y = freq_counts.get(Y)
                if sup_count_y is None:
                    # Y też może nie być częste (zależy od min_support), wtedy pomijamy metryki zależne od support(Y)
                    # Możesz też tu policzyć support(Y) inaczej, ale to wymaga ponownego przejścia po transakcjach.
                    continue

                support_y = sup_count_y / n_transactions

                lift = confidence / support_y if support_y > 0 else float("inf")
                leverage = support_xy - (support_x * support_y)

                rules.append(
                    Rule(
                        antecedent=X,
                        consequent=Y,
                        support=support_xy,
                        confidence=confidence,
                        lift=lift,
                        leverage=leverage,
                    )
                )

    # sort domyślny: najpierw confidence, potem lift, potem support
    rules.sort(key=lambda r: (r.confidence, r.lift, r.support), reverse=True)
    return rules


def print_top_rules(rules: List[Rule], top_n: int = 20) -> None:
    for i, r in enumerate(rules[:top_n], start=1):
        print(
            f"{i:02d}. {sorted(r.antecedent)} -> {sorted(r.consequent)} | "
            f"support={r.support:.3f} conf={r.confidence:.3f} lift={r.lift:.3f} lev={r.leverage:.3f}"
        )

def augment_transactions(
    transactions: List[Itemset],
    n_new: int = 1_000_000,
    p_add: float = 0.35,
    p_remove: float = 0.35,
    max_add: int = 2,
    max_remove: int = 2,
    seed: int = 123,
) -> List[Itemset]:
    """
    Tworzy n_new nowych transakcji na bazie istniejących:
    - losuje bazową transakcję
    - usuwa/dodaje kilka losowych elementów z pewnym prawdopodobieństwem
    Zwraca powiększoną listę: oryginalne + nowe.
    """
    rng = random.Random(seed)

    # uniwersum produktów
    all_items: List[Item] = sorted({item for t in transactions for item in t})

    new_transactions: List[Itemset] = []
    base_n = len(transactions)

    for _ in range(n_new):
        base = transactions[rng.randrange(base_n)]
        s: Set[Item] = set(base)

        # usuń
        if s and rng.random() < p_remove:
            k = rng.randint(1, min(max_remove, len(s)))
            for item in rng.sample(list(s), k):
                s.remove(item)

        # dodaj
        if rng.random() < p_add:
            # dodaj tylko te, których nie ma
            candidates = [x for x in all_items if x not in s]
            if candidates:
                k = rng.randint(1, min(max_add, len(candidates)))
                for item in rng.sample(candidates, k):
                    s.add(item)

        new_transactions.append(frozenset(s))

    return transactions + new_transactions


if __name__ == "__main__":
    MIN_SUPPORT = 0.10
    MIN_CONFIDENCE = 0.55

    transactions = load_market_csv("market.csv")
    # print(transactions)

    # np. próg 30% (jak rozważałeś)
    freq = apriori(transactions, min_support=MIN_SUPPORT)

    # wypisz 10 najczęstszych zbiorów (po liczności)
    # top = sorted(freq.items(), key=lambda kv: kv[1], reverse=True)[:100]
    # for itemset, cnt in top:
    #     print(cnt, sorted(itemset))

    n = len(transactions)

    rules = generate_association_rules(
        freq_counts=freq,
        n_transactions=n,
        min_support=MIN_SUPPORT,      
        min_confidence=MIN_CONFIDENCE,   
    )

    print_top_rules(rules, top_n=100)

    t0 = time.perf_counter()
    transactions_big = augment_transactions(transactions, n_new=1_000_000)
    t1 = time.perf_counter()

    print(f"Generowanie danych: {t1 - t0:.2f} s")
    print(f"Liczba transakcji: {len(transactions_big)}")

    # Apriori
    t2 = time.perf_counter()
    freq_big = apriori(transactions_big, min_support=MIN_SUPPORT)  # próg ustaw jak wcześniej
    t3 = time.perf_counter()

    print(f"Apriori na big data: {t3 - t2:.2f} s")
    print(f"Liczba częstych zbiorów: {len(freq_big)}")


01. ['Bacon', 'Onion'] -> ['Banana'] | support=0.131 conf=0.701 lift=1.564 lev=0.047
02. ['Olive', 'ShavingFoam'] -> ['Banana'] | support=0.119 conf=0.688 lift=1.534 lev=0.041
03. ['Banana', 'Butter'] -> ['Bacon'] | support=0.121 conf=0.675 lift=1.565 lev=0.044
04. ['Butter', 'ShavingFoam'] -> ['Bacon'] | support=0.127 conf=0.670 lift=1.555 lev=0.045
05. ['Bacon', 'Butter'] -> ['Cheese'] | support=0.138 conf=0.667 lift=1.502 lev=0.046
06. ['Butter', 'Carrot'] -> ['Cheese'] | support=0.106 conf=0.653 lift=1.472 lev=0.034
07. ['Butter', 'Egg'] -> ['Cheese'] | support=0.106 conf=0.653 lift=1.472 lev=0.034
08. ['Bread', 'ShavingFoam'] -> ['Bacon'] | support=0.121 conf=0.651 lift=1.511 lev=0.041
09. ['Butter', 'Onion'] -> ['Cheese'] | support=0.116 conf=0.651 lift=1.465 lev=0.037
10. ['Bacon', 'Egg'] -> ['Cheese'] | support=0.131 conf=0.649 lift=1.462 lev=0.042
11. ['Butter', 'Meat'] -> ['Cheese'] | support=0.106 conf=0.645 lift=1.452 lev=0.033
12. ['Apple', 'Bacon'] -> ['Banana'] | support

Komentarz do zadań.

Jak sensownie ustawić progi support i confidence dla reguł asocjacyjnych?

Trzeba analizować dane. Mozna wybierac reguły na podstawie samego supporta, albo samego confidence. Korzysta się jeszcze z iloczyny tych dwóch wskaźników.

Jak wybrać z nich te najbardziej wartościowe? Może warto użyć lift lub leverage? Może masz pomysły na inne metryki oceniające reguły asocjacyjne?

Generalnie to warto uzyć lift i leverage, jest jeszcze jeden wskaźnik taki jak conviction.
Lift i leverage sprawdzają, czy te zasady są zalezne od siebie w jakim stopni. Najlepiej, zeby lift był większy od 1, a dokładnie rózny od 1. Leverage natomiast fajnie jakby był dodatni.

https://en.wikipedia.org/wiki/Association_rule_learning